# Step 06 — ComBat, federated and exact

**Data type: RNA_array** (GSE65391). **Reads:** `step05_site_{A,B,C}.rds`.
**Writes:** `step06_site_{A,B,C}.rds`.

ComBat (Johnson et al. 2007) removes a batch effect gene by gene: it estimates each batch's shift and
scale, and shrinks those estimates towards the batch's average over all genes (empirical Bayes), which
keeps them stable when a batch is small.

**ComBat needs very little from the other sites.** It has two kinds of quantity:

| quantity | computed from | where |
|---|---|---|
| grand mean of each gene | every sample | needs all sites, but only their per-gene count, sum and sum of squares |
| pooled within-site variance of each gene | every sample | the same three numbers |
| each site's shift and scale per gene | that site's samples | at the site |
| empirical Bayes priors | that site's genes | at the site |

So one round of summaries is enough. Each site sends three numbers per gene. The coordinator returns
the grand mean and the pooled variance. Each site finishes ComBat on its own. `mod = NULL`: ComBat is
told nothing about disease.

In [1]:
source("../src/paths.R")
source("../src/federation.R")
start_log("06")

## Round 1: each site sends n, sum and sum of squares per gene

In [2]:
msgs <- lapply(setNames(SITES, SITES), function(s) {
  d <- readRDS(site_file("05", s))
  send(site_suff(d$E), s, "per-gene n, sum, sum of squares", ncol(d$E))
})
sapply(msgs, function(m) c(samples = m$n, genes = length(m$sum)))

,A,B,C
samples,338,280,354
genes,27984,27984,27984


## The coordinator returns two vectors

In [3]:
grand_mean <- pool_suff(msgs)$mean
var_pooled <- pool_within_var(msgs)
c(genes = length(grand_mean), median_pooled_sd = round(median(sqrt(var_pooled)), 3))

genes median_pooled_sd 
       27984.000            0.294

## Round 2: each site adjusts its own data

In [4]:
for (s in SITES) {
  d   <- readRDS(site_file("05", s))
  fit <- site_combat_adjust(d$E, grand_mean, var_pooled)
  d$E <- fit$X
  d$combat <- data.frame(gene = rownames(d$E), gamma_star = fit$gamma_star, delta_star = fit$delta_star)
  saveRDS(d, site_file("06", s))
  cat(sprintf("site %s adjusted: median |shift removed| %.3f SD units\n", s, median(abs(fit$gamma_star))))
}

site A adjusted: median |shift removed| 0.715 SD units
site B adjusted: median |shift removed| 0.796 SD units
site C adjusted: median |shift removed| 0.699 SD units


## Oracle — possible only because this is a simulation

Is federated ComBat the same as ComBat run on all samples in one place? We pool the three sites' data
here, and only here, to check. The result of this cell is used for nothing else.

In [5]:
suppressMessages(library(sva))
planted <- lapply(SITES, function(s) readRDS(site_file("05", s))$E)
fed     <- do.call(cbind, lapply(SITES, function(s) readRDS(site_file("06", s))$E))
central <- suppressMessages(ComBat(do.call(cbind, planted), rep(SITES, sapply(planted, ncol)),
                                   mod = NULL, par.prior = TRUE, prior.plots = FALSE))
gap <- max(abs(fed - central))
stopifnot(gap < 1e-8)
cat(sprintf("GATE PASSED: federated ComBat equals sva::ComBat to %.1e\n", gap))
rm(planted, fed, central); invisible(gc())

GATE PASSED: federated ComBat equals sva::ComBat to 1.7e-10


## Findings

Federated ComBat is ComBat. One round of per-gene summaries from each site, then local work, gives
the same corrected values as `sva::ComBat` on all samples together, to within numerical rounding.